In [1]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances

In [2]:
active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772}
active_lr_event_ids = {'769': 769, '770': 770}
unknown_event_id = {'783': 783} 

In [3]:
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2a'

In [4]:
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [5]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [ ]:
# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.
sfreq = 250
tmin = 0.5       # Epoch start at cue onset.
# Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
tmax = 3.5  # This gives 4.0 seconds.
filter_order = 50

In [ ]:
def twofour_crosssession(n_classes):

    if(n_classes==2):
        event_ids = active_lr_event_ids
    else:
        event_ids = active_all_event_ids
    

    train_active_X = []         # List to hold numpy arrays with shape (n_trials, 22, 1001) per subject.
    train_active_y = []         # List to hold event labels per subject.
    train_active_metadata = []  # List to hold event metadata per subject.

    # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
    for subj in range(1, 10):
        filename = os.path.join(data_dir, f'A{subj:02d}T.gdf')
        
        # Read the GDF file (using preload=True to load data into memory).
        train_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        # Retain only EEG channels (22 channels) and exclude EOG channels.
        train_raw.pick_types(eeg=True, eog=False)
        
        # Extract events corresponding only to the four desired types.
        train_active_events, _ = mne.events_from_annotations(train_raw, event_id=event_ids)
        
        # Create epochs from tmin to tmax.
        train_active_epochs = mne.Epochs(train_raw, train_active_events, event_id=event_ids, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)
        
        # Get the epoch data (num_epochs x 22 channels x 1001 samples).
        train_active_data = train_active_epochs.get_data()
        
        # Print the number of extracted epochs to verify
        print(f"Subject {subj}: Epoch data shape {train_active_data.shape}")
        
        # Sampling frequency from raw.info (should be 250).
        fs = int(train_raw.info['sfreq'])
        # print(fs)
        n_trials, n_channels, n_times = train_active_data.shape
        train_active_filtered_data = np.empty_like(train_active_data)
        
        # Apply the causal bandpass filter channel‐wise for each trial.
        for trial in range(n_trials):
            for ch in range(n_channels):
                train_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    train_active_data[trial, ch, :],
                    lowcut=8,    # Lower bound of sensorimotor rhythm.
                    highcut=30,  # Upper bound of sensorimotor rhythm.
                    fs=fs,
                    order=filter_order     # Lower order for a smoother causal filter.
                )
        # Apply the causal bandpass filter channel‐wise for each trial.
        # for trial in range(n_trials):
        #     for ch in range(n_channels):
        #         train_active_filtered_data[trial, ch, :] = train_active_data[trial, ch, :]
        
        # Append the processed data, labels, and event metadata.
        train_active_X.append(train_active_filtered_data)
        train_active_y.append(train_active_epochs.events[:, 2])  # The third column holds the event code.
        train_active_metadata.append(train_active_epochs.events)

    print("Loaded data for", len(train_active_X), "subjects.")

    eval_active_X = []         
    eval_active_y = []        
    eval_active_metadata = [] 

    # Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
    for subj in range(1, 10):
        filename = os.path.join(data_dir, f'A{subj:02d}E.gdf')
        mat_data = loadmat(f'/home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels/A{subj:02d}E.mat')
        true_y =  np.array(mat_data['classlabel'], dtype=np.int64).reshape(288,) + 768
        # Read the GDF file (using preload=True to load data into memory).
        eval_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
        # Retain only EEG channels (22 channels) and exclude EOG channels.
        eval_raw.pick_types(eeg=True, eog=False)
        
        # Extract events corresponding only to the four desired types.
        eval_active_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id)
        
        # Create epochs from tmin to tmax.
        eval_active_epochs = mne.Epochs(eval_raw, eval_active_events, event_id=unknown_event_id, tmin=tmin, tmax=tmax,
                            baseline=None, preload=True, verbose=False)
        
        # Get the epoch data (num_epochs x 22 channels x 1001 samples).
        eval_active_data = eval_active_epochs.get_data()
        
        # Print the number of extracted epochs to verify
        print(f"Subject {subj}: Epoch data shape {eval_active_data.shape}")
        
        # Sampling frequency from raw.info (should be 250).
        fs = int(eval_raw.info['sfreq'])
        # print(fs)
        n_trials, n_channels, n_times = eval_active_data.shape
        eval_active_filtered_data = np.empty_like(eval_active_data)
        
        # Apply the causal bandpass filter channel‐wise for each trial.
        for trial in range(n_trials):
            for ch in range(n_channels):
                eval_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                    eval_active_data[trial, ch, :],
                    lowcut=8,    # Lower bound of sensorimotor rhythm.
                    highcut=30,  # Upper bound of sensorimotor rhythm.
                    fs=fs,
                    order=filter_order     # Lower order for a smoother causal filter.
                )
        
        # for trial in range(n_trials):
        #     for ch in range(n_channels):
        #         eval_active_filtered_data[trial, ch, :] = eval_active_data[trial, ch, :]
                    
        
        # Append the processed data, labels, and event metadata.
        eval_active_X.append(eval_active_filtered_data)
        eval_active_y.append(true_y)  # The third column holds the event code.
        eval_active_metadata.append(eval_active_epochs.events)

    print("Loaded data for", len(eval_active_X), "subjects.")

    if(n_classes==2):
        eval_active_X = [x[np.isin(y, [769, 770])] for x, y in zip(eval_active_X, eval_active_y)]
        eval_active_y = [y[np.isin(y, [769, 770])] for y in eval_active_y]
    
    train_active_y = encode_labels(train_active_y)
    eval_active_y = encode_labels(eval_active_y)

    # train_aligned = []
    # eval_aligned = []
    # def EA(X):
    #     covs = np.array([trial @ trial.T for trial in X])  # shape [n_trials, n_channels, n_channels]
    #     mean_cov = np.mean(covs, axis=0)
    #     # Eigen-decomposition of mean_cov
    #     eigvals, eigvecs = np.linalg.eigh(mean_cov)
    #     D_inv_sqrt = np.diag(eigvals**(-0.5))
    #     R_inv_sqrt = eigvecs @ D_inv_sqrt @ eigvecs.T

    #     # Align each trial
    #     xaligned = np.array([R_inv_sqrt @ trial for trial in X])
    #     return xaligned

    # for i in range(9):
    #     train_aligned.append(EA(train_active_X[i]))
    #     eval_aligned.append(EA(eval_active_X[i]))
    
    n_components = 4
    csp = CSP(n_components=n_components, reg=None, log=True, norm_trace=False)
    lda = LinearDiscriminantAnalysis()

    accuracies = []

    for subj_idx in range(len(train_active_X)):
        # print(subj_idx)
        # Split data into train/test using leave-one-subject-out
        X_test = eval_active_X[subj_idx]
        y_test = eval_active_y[subj_idx]
        
        # Concatenate data from other subjects
        X_train = train_active_X[subj_idx]
        y_train = train_active_y[subj_idx]

        # Create pipeline
        pipeline = Pipeline([
            ('CSP', CSP(n_components=n_components, reg=None, log=True, norm_trace=False)),
            ('LDA', LinearDiscriminantAnalysis())
        ])

        # Fit and predict
        pipeline.fit(X_train, y_train)
        accuracy = pipeline.score(X_test, y_test)
        accuracies.append(accuracy)
    
    for subj_idx in range(len(train_active_X)):
        print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

    print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
    

        
    

In [8]:
twofour_crosssession(2)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 1: Epoch data shape (144, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 2: Epoch data shape (144, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 7: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 8: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Epoch data shape (288, 22, 751)
Loaded data for 9 subjects.
Computing rank from data with rank=None
    Using tolerance 7e-05 (2.2e-16 eps * 22 dim * 1.4e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 6e-05 (2.2e-16 eps * 22 dim * 1.2e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 6.7e-05 (2.2e-16 eps * 22 dim * 1.4e+10  max si

In [9]:
twofour_crosssession(4)

/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 1: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770', '771', '772']
Subject 2: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 7: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 8: Epoch data shape (288, 22, 751)


/home/vishwa/anaconda3/envs/eeg_proj/lib/python3.10/contextlib.py:142: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 9: Epoch data shape (288, 22, 751)
Loaded data for 9 subjects.
Computing rank from data with rank=None
    Using tolerance 0.0001 (2.2e-16 eps * 22 dim * 2.1e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Estimating class=3 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 8.5e-05 (2.2e-16 eps * 22 dim * 1.7e+10  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
